# MITS: DKT Training on ASSISTments

Train LSTM-based Deep Knowledge Tracing model.

**Dataset**: ASSISTments 2009-2010 skill-builder (~346K attempts, 4K students, 123 skills)

**Target**: AUC > 0.75 on test set

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score, accuracy_score
import json
import time

print(f"PyTorch: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load Data

In [ ]:
DATA_DIR = Path("data/training")

# Load preprocessed data (run prepare_assistments.py first)
train_data = torch.load(DATA_DIR / "assistments_train.pt")
val_data = torch.load(DATA_DIR / "assistments_val.pt")
test_data = torch.load(DATA_DIR / "assistments_test.pt")

with open(DATA_DIR / "assistments_skill_map.json") as f:
    skill_map = json.load(f)

NUM_SKILLS = len(skill_map)
print(f"Skills: {NUM_SKILLS}")
print(f"Train: {len(train_data['skills'])} sequences")
print(f"Val: {len(val_data['skills'])} sequences")
print(f"Test: {len(test_data['skills'])} sequences")

In [ ]:
def make_dataloader(data, batch_size=64, shuffle=True):
    dataset = TensorDataset(data["skills"], data["corrects"], data["lengths"])
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

BATCH_SIZE = 64
train_loader = make_dataloader(train_data, BATCH_SIZE, shuffle=True)
val_loader = make_dataloader(val_data, BATCH_SIZE, shuffle=False)
test_loader = make_dataloader(test_data, BATCH_SIZE, shuffle=False)

## 2. DKT Model

In [ ]:
class DKT(nn.Module):
    def __init__(self, num_skills, hidden_size=64, num_layers=1, dropout=0.2, embed_size=32):
        super().__init__()
        self.num_skills = num_skills
        self.hidden_size = hidden_size
        
        # Input: skill_id * 2 + correctness -> 2 * num_skills possible inputs
        self.embedding = nn.Embedding(num_skills * 2, embed_size, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
        )
        self.dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, num_skills)
    
    def forward(self, skills, corrects, lengths):
        # Encode input: skill_id * 2 + correctness
        # Clamp to valid range
        inputs = (skills * 2 + corrects).clamp(0, self.num_skills * 2 - 1)
        
        # Mask padding
        mask = skills >= 0
        inputs = inputs * mask.long()
        
        x = self.embedding(inputs)  # (B, T, E)
        x, _ = self.lstm(x)         # (B, T, H)
        x = self.dropout(x)
        logits = self.output(x)     # (B, T, num_skills)
        
        return logits, mask


# Hyperparameters
HIDDEN_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.2
EMBED_SIZE = 32
LR = 1e-3
EPOCHS = 20

model = DKT(NUM_SKILLS, HIDDEN_SIZE, NUM_LAYERS, DROPOUT, EMBED_SIZE).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

## 3. Training

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(reduction="none")


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    n_samples = 0
    
    for skills, corrects, lengths in loader:
        skills = skills.to(device)
        corrects = corrects.to(device)
        lengths = lengths.to(device)
        
        logits, mask = model(skills, corrects, lengths)
        
        # Target: correctness at next time step
        # Use skills at t+1 to select the relevant logit
        target_skills = skills[:, 1:]  # Next skill
        target_correct = corrects[:, 1:].float()  # Next correctness
        pred_logits = logits[:, :-1]  # Predictions from current step
        target_mask = mask[:, 1:]  # Mask for valid targets
        
        # Gather predictions for the target skill
        target_skills_clamped = target_skills.clamp(0, NUM_SKILLS - 1)
        pred = pred_logits.gather(2, target_skills_clamped.unsqueeze(-1)).squeeze(-1)
        
        # Compute loss only on valid positions
        loss = criterion(pred, target_correct)
        loss = (loss * target_mask.float()).sum() / target_mask.float().sum().clamp(min=1)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item() * target_mask.float().sum().item()
        n_samples += target_mask.float().sum().item()
    
    return total_loss / max(n_samples, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_targets = []
    
    for skills, corrects, lengths in loader:
        skills = skills.to(device)
        corrects = corrects.to(device)
        
        logits, mask = model(skills, corrects, lengths.to(device))
        
        target_skills = skills[:, 1:].clamp(0, NUM_SKILLS - 1)
        target_correct = corrects[:, 1:].float()
        target_mask = mask[:, 1:]
        
        pred = logits[:, :-1].gather(2, target_skills.unsqueeze(-1)).squeeze(-1)
        pred = torch.sigmoid(pred)
        
        valid = target_mask.bool().cpu().numpy().flatten()
        all_preds.extend(pred.cpu().numpy().flatten()[valid])
        all_targets.extend(target_correct.cpu().numpy().flatten()[valid])
    
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    auc = roc_auc_score(all_targets, all_preds)
    acc = accuracy_score(all_targets, (all_preds > 0.5).astype(int))
    
    return auc, acc

In [ ]:
# Training loop
best_val_auc = 0
best_epoch = 0
history = []

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_auc, val_acc = evaluate(model, val_loader)
    
    elapsed = time.time() - start
    history.append({"epoch": epoch, "train_loss": train_loss, "val_auc": val_auc, "val_acc": val_acc})
    
    print(f"Epoch {epoch:2d}/{EPOCHS} | Loss: {train_loss:.4f} | Val AUC: {val_auc:.4f} | Val Acc: {val_acc:.4f} | {elapsed:.1f}s")
    
    # Save best model
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_epoch = epoch
        torch.save({
            "model_state_dict": model.state_dict(),
            "num_skills": NUM_SKILLS,
            "hidden_size": HIDDEN_SIZE,
            "num_layers": NUM_LAYERS,
            "embed_size": EMBED_SIZE,
            "val_auc": val_auc,
            "epoch": epoch,
        }, "data/models/dkt_pretrained.pt")
        print(f"  -> New best! Saved checkpoint.")

print(f"\nBest: epoch {best_epoch}, AUC = {best_val_auc:.4f}")

## 4. Test Evaluation

In [ ]:
# Load best checkpoint
checkpoint = torch.load("data/models/dkt_pretrained.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])

test_auc, test_acc = evaluate(model, test_loader)
print(f"Test AUC: {test_auc:.4f}")
print(f"Test Acc: {test_acc:.4f}")
print(f"Target AUC > 0.75: {'PASS' if test_auc > 0.75 else 'FAIL'}")

## 5. Training History

In [ ]:
# Plot training history
try:
    import matplotlib.pyplot as plt
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    epochs_list = [h["epoch"] for h in history]
    
    ax1.plot(epochs_list, [h["train_loss"] for h in history])
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Training Loss")
    ax1.set_title("Training Loss")
    
    ax2.plot(epochs_list, [h["val_auc"] for h in history], label="AUC")
    ax2.plot(epochs_list, [h["val_acc"] for h in history], label="Accuracy")
    ax2.axhline(y=0.75, color="r", linestyle="--", label="Target AUC")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Score")
    ax2.set_title("Validation Metrics")
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig("data/models/dkt_training_history.png", dpi=150)
    plt.show()
except ImportError:
    print("matplotlib not available, skipping plot")